# Tutorial - AIME 2026: From LLMs to DNA
## 🧬 Notebook 2. Genomic Sequence Embeddings & Downstream Tasks  

Welcome to the second phase of our tutorial! In Notebook 1, we successfully built and trained a custom Genomic BERT model from scratch.

In this standalone environment, we will load our tokenizer and trained weights, fetch the real Hugging Face genomic dataset, and generate dense mathematical embeddings for the unseen **Test Set** (Chromosome 22).


## 1. Environment Setup & GitHub Tokenizer
This cell prepares the PyTorch environment and loads the selected tokenizer directly from the GitHub `models/` folder. Change `MODEL_OPTION` to switch between character and k-mer tokenization strategies.

In [ ]:
!pip install datasets transformers tqdm torch -q

import os
import sys
import re
import urllib.request
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import PreTrainedTokenizerFast, BertConfig, BertForMaskedLM
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Hardware Allocation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target execution hardware detected: {str(device).upper()}")

# Select tokenizer / model option from GitHub
MODEL_OPTION = "kmer_k6_s3"  # options: character, kmer_k3_s1, kmer_k3_s3, kmer_k4_s1, kmer_k4_s2, kmer_k5_s1, kmer_k5_s2, kmer_k6_s1, kmer_k6_s3, kmer_k8_s4

REPO_RAW = "https://raw.githubusercontent.com/pabloarozarena/aime2026-T7-genomic-llms/main/models"
TOKENIZER_URL = f"{REPO_RAW}/{MODEL_OPTION}/tokenizer"
WEIGHTS_URL = f"{REPO_RAW}/{MODEL_OPTION}/weights/model_state_dict.pt"

TOKENIZATION_STYLE = "character" if MODEL_OPTION == "character" else "kmer"

if TOKENIZATION_STYLE == "kmer":
    match = re.search(r"kmer_k(\d+)_s(\d+)", MODEL_OPTION)
    K_SIZE = int(match.group(1))
    STRIDE = int(match.group(2))
else:
    K_SIZE = None
    STRIDE = None

# Sequence formatter

def kmer_string_splitter(sequence, k, stride):
    return " ".join([sequence[i:i+k] for i in range(0, len(sequence) - k + 1, stride)])


def format_sequence(sequence):
    sequence = sequence.upper()
    if TOKENIZATION_STYLE == "kmer":
        return kmer_string_splitter(sequence, K_SIZE, STRIDE)
    return " ".join(list(sequence))

# Download and load tokenizer from GitHub
tokenizer_dir = f"github_model_files/{MODEL_OPTION}/tokenizer"
os.makedirs(tokenizer_dir, exist_ok=True)

urllib.request.urlretrieve(f"{TOKENIZER_URL}/tokenizer.json", f"{tokenizer_dir}/tokenizer.json")
urllib.request.urlretrieve(f"{TOKENIZER_URL}/tokenizer_config.json", f"{tokenizer_dir}/tokenizer_config.json")

hf_tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_dir)

print(f"✓ Loaded tokenizer: {MODEL_OPTION}")
print(f"✓ Tokenization style: {TOKENIZATION_STYLE}")
print(f"✓ Vocabulary size: {len(hf_tokenizer):,}")
if TOKENIZATION_STYLE == "kmer":
    print(f"✓ k-mer size: {K_SIZE} | stride: {STRIDE}")


## 2. Network Generation and Weight Mounting
Here we reconstruct the architectural blueprint of our Genomic BERT (`BertForMaskedLM` with 128 hidden dimensions) and load the matching weights for the selected tokenizer/model option from GitHub.

In [ ]:
# Re-instantiate the architectural blueprint shell
config = BertConfig(
    vocab_size=len(hf_tokenizer),
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=2,
    intermediate_size=512,
    max_position_embeddings=512
)

print("Constructing empty structural model shell...")
model = BertForMaskedLM(config)

# Download and load selected weights from GitHub
weights_path = f"github_model_files/{MODEL_OPTION}/weights/model_state_dict.pt"
os.makedirs(os.path.dirname(weights_path), exist_ok=True)

urllib.request.urlretrieve(WEIGHTS_URL, weights_path)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)

# Mount to GPU and lock weights for inference
model.to(device)
model.eval()
print("\n" + "="*60)
print(f"✓ SUCCESS: {MODEL_OPTION} Genomic BERT is online and ready for inference!")
print("="*60)


## 3. Test Set Tokenization & DataLoader Construction
We load the Chromosome 22 dataset and tokenize it using the selected character or k-mer tokenizer.

In [ ]:
import pandas as pd

chr22_url = "https://raw.githubusercontent.com/pabloarozarena/aime2026-T7-genomic-llms/main/data/chr22.csv"

chr22_df = pd.read_csv(chr22_url)

pd.DataFrame(chr22_df)

In [ ]:
from datasets import Dataset
from torch.utils.data import DataLoader

hf_test_raw = Dataset.from_pandas(chr22_df, preserve_index=False)

print(f"Loaded GitHub Dataset containing {len(hf_test_raw):,} rows.")

def test_batch_mapper(batch):
    processed_strings = [format_sequence(seq) for seq in batch["seq"]]

    return hf_tokenizer(
        processed_strings,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

print("Executing test sequence vector alignment protocol...")

tokenized_test = hf_test_raw.map(
    test_batch_mapper,
    batched=True,
    batch_size=2000,
    desc="Tokenizing Test Set"
)

metadata_columns = ["chrom", "start", "end", "gc_content", "seq"]
clean_test_dataset = tokenized_test.remove_columns(metadata_columns)

clean_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_loader = DataLoader(
    clean_test_dataset,
    batch_size=256,
    shuffle=False
)

print(f"✓ Evaluation Stream active across {len(test_loader)} batches.")

## 4. Latent Feature Vector Extraction (Mean Pooling)
We pass the tokenized sequences through our frozen BERT backbone. By averaging the non-padded hidden states of the final transformer layer, we project every 512bp DNA window into a dense, 128-dimensional latent vector.

In [ ]:
# Bypasses the Hugging Face vision bug when dealing with tensors
sys.modules.pop("torchvision", None)

print("Commencing embedding matrix generation protocol...")
all_embeddings = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Extracting Sequence Vectors"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass targeting the base BERT architecture
        transformer_outputs = model.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = transformer_outputs.last_hidden_state

        # Perform MEAN POOLING
        input_mask_expanded = (
            attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        )

        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        batch_mean_embeddings = sum_embeddings / sum_mask

        # Offload calculated vectors to CPU
        all_embeddings.append(batch_mean_embeddings.cpu().numpy())

test_embeddings_matrix = np.vstack(all_embeddings)
print(f"\n✓ EMBEDDING EXTRACTION SUCCESSFUL! Matrix Shape: {test_embeddings_matrix.shape}")

## 5. PCA Projection of Latent Space
To understand the macroscopic rules learned by our custom model during its pre-training phase, we reduce the 128-dimensional embeddings to a 2D linear plane using Principal Component Analysis (PCA).

In [ ]:
from sklearn.decomposition import PCA

print("Executing PCA dimension reduction protocol (128 -> 2)...")

pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(test_embeddings_matrix)

var_pc1 = pca.explained_variance_ratio_[0] * 100
var_pc2 = pca.explained_variance_ratio_[1] * 100

print(f"✓ PCA complete!")
print(f"PC1 explains: {var_pc1:.2f}% of the total variance")
print(f"PC2 explains: {var_pc2:.2f}% of the total variance")

plt.figure(figsize=(9, 6))
plt.scatter(pca_result[:, 0], pca_result[:, 1], alpha=0.5, s=3, c='gray')
plt.title("Base PCA Projection of Genomic Embeddings", fontweight="bold")
plt.xlabel(f"PC1 ({var_pc1:.1f}%)")
plt.ylabel(f"PC2 ({var_pc2:.1f}%)")
plt.grid(True, alpha=0.2)
plt.show()

## 6. PCA Manifold by GC Content
Here, we map the inherent GC-content percentage onto the generated PCA manifold. This visually tests whether our model’s pre-trained attention heads independently recognized base-pair saturation as a dominant geometric feature.

In [ ]:
print("Rendering PCA by GC Content Gradient...")

gc_percentages = np.array(hf_test_raw["gc_content"])

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    pca_result[:, 0],
    pca_result[:, 1],
    c=gc_percentages,
    cmap="viridis",
    alpha=0.7,
    s=3,
)

plt.title("Chromosome 22 PCA: GC Content Gradient", fontsize=14, fontweight="bold")
plt.xlabel(f"Principal Component 1 ({var_pc1:.1f}%)", fontweight="bold")
plt.ylabel(f"Principal Component 2 ({var_pc2:.1f}%)", fontweight="bold")
plt.grid(True, alpha=0.1)

cbar = plt.colorbar(scatter)
cbar.set_label("GC Content (%)", rotation=270, labelpad=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. PCA Manifold by Feature Geography (Exon vs. Intron)
Using our binary targets (0 for Exons, 1 for Introns), we apply a high-contrast overlay to the PCA plane. Distinct spatial separation here proves the model natively distinguishes functional coding zones from structural non-coding zones.

In [ ]:
print("Rendering PCA by Genomic Feature Type...")

labels = np.array(hf_test_raw["label"])

plt.figure(figsize=(10, 7))

plt.scatter(
    pca_result[labels == 0, 0], pca_result[labels == 0, 1],
    c="#FF6B6B", alpha=0.6, s=3, label="Exons (Label 0)"
)
plt.scatter(
    pca_result[labels == 1, 0], pca_result[labels == 1, 1],
    c="#4D96FF", alpha=0.6, s=3, label="Introns (Label 1)"
)

plt.title("Chromosome 22 PCA: Global Exon vs. Intron Separation", fontsize=14, fontweight="bold")
plt.xlabel(f"Principal Component 1 ({var_pc1:.1f}%)", fontweight="bold")
plt.ylabel(f"Principal Component 2 ({var_pc2:.1f}%)", fontweight="bold")
plt.grid(True, alpha=0.1)
plt.legend(loc="upper right", markerscale=5, fontsize=11, frameon=True)

plt.tight_layout()
plt.show()

## 8. Downstream Classification (Gradient Boosting Probe)
Finally, we apply a non-linear feature extractor (HistGBDT) over our native 128-dimensional space. The code stratifies the evaluation targets (80/10/10) and outputs precision metrics, ROC-AUC limits, and a stylized confusion matrix to assess the model's predictive limits without active fine-tuning.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Preparing stratifed splits and training downstream classifier...")

# Stratified Data Split
X = test_embeddings_matrix
y = labels

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Train Non-Linear Probe
nl_classifier = HistGradientBoostingClassifier(
    max_iter=150, learning_rate=0.1, l2_regularization=1.0, random_state=42
)
nl_classifier.fit(X_train, y_train)

# Predict and Display Terminal Metrics
nl_test_probs = nl_classifier.predict_proba(X_test)[:, 1]
nl_test_preds = nl_classifier.predict(X_test)

print("\n" + "=" * 20 + " FINAL TEST METRICS " + "=" * 20)
print(f"Final Test ROC-AUC Score: {roc_auc_score(y_test, nl_test_probs):.4f}")
print(classification_report(y_test, nl_test_preds, target_names=["Exon", "Intron"]))
print("=" * 60)

# Render Confusion Matrix Display
print("\nGenerating Confusion Matrix...")
cm = confusion_matrix(y_test, nl_test_preds)

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    nl_test_preds,
    display_labels=["Exon", "Intron"],
    cmap="Blues",
    ax=ax,
    colorbar=False
)

plt.title("Downstream Test Set Confusion Matrix", fontsize=12, fontweight="bold", pad=10)
plt.grid(False)
plt.tight_layout()
plt.show()